In [11]:
# Import libraries
import pandas as pd
import numpy as np
import sqlite3
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [32]:
# 1. Load the Datasets
companies = pd.read_excel('companies.xlsx', header=1)
companies.columns = [str(c).strip().lower() for c in companies.columns]
pnl = pd.read_excel('profitandloss.xlsx', header=1)
pnl.columns = [str(c).strip().lower() for c in pnl.columns]
bs = pd.read_excel('balancesheet.xlsx', header=1)
bs.columns = [str(c).strip().lower() for c in bs.columns]
cf = pd.read_excel('cashflow.xlsx', header=1)
cf.columns = [str(c).strip().lower() for c in cf.columns]
analysis = pd.read_excel('analysis.xlsx', header=1)
analysis.columns = [str(c).strip().lower() for c in analysis.columns]
docs = pd.read_excel('documents.xlsx', header=1)
docs.columns = [str(c).strip().lower() for c in docs.columns]
pros_cons = pd.read_excel('prosandcons.xlsx', header=1)
pros_cons.columns = [str(c).strip().lower() for c in pros_cons.columns]

In [33]:
# 3. Create SQLite Database & Load Tables
conn = sqlite3.connect('nifty100.db')

companies.to_sql('companies', conn, if_exists='replace', index=False)
pnl.to_sql('profitandloss', conn, if_exists='replace', index=False)
bs.to_sql('balancesheet', conn, if_exists='replace', index=False)
cf.to_sql('cashflow', conn, if_exists='replace', index=False)
analysis.to_sql('analysis', conn, if_exists='replace', index=False)
docs.to_sql('documents', conn, if_exists='replace', index=False)
pros_cons.to_sql('prosandcons', conn, if_exists='replace', index=False)

print("Module 1 Complete: ETL Pipeline ran successfully and data is saved to nifty100.db!")

Module 1 Complete: ETL Pipeline ran successfully and data is saved to nifty100.db!


In [34]:
# Query merged P&L and Balance Sheet data
query = """
SELECT
    p.company_id, p.year, p.sales, p.operating_profit, p.net_profit,
    b.equity_capital, b.reserves, b.borrowings, b.total_liabilities, b.total_assets
FROM profitandloss p
JOIN balancesheet b ON p.company_id = b.company_id AND p.year = b.year
"""
financials = pd.read_sql(query, conn)

In [35]:
# Compute Core KPIs
# 1. Operating Profit Margin (%)
financials['OPM_pct'] = (financials['operating_profit'] / financials['sales']) * 100

# 2. Net Worth & Return on Equity (ROE %)
financials['net_worth'] = financials['equity_capital'] + financials['reserves']
financials['ROE_pct'] = (financials['net_profit'] / financials['net_worth']) * 100

# 3. Debt to Equity Ratio
financials['Debt_to_Equity'] = financials['borrowings'] / financials['net_worth']

# Save Computed KPIs back to the database
financials.to_sql('financial_ratios', conn, if_exists='replace', index=False)
print("Module 2 Complete: Financial Ratio Engine successfully computed KPIs.")
display(financials[['company_id', 'year', 'OPM_pct', 'ROE_pct', 'Debt_to_Equity']].head())

Module 2 Complete: Financial Ratio Engine successfully computed KPIs.


,company_id,year,OPM_pct,ROE_pct,Debt_to_Equity
0,ABB,Dec 2012,12.220206,22.411128,0.0
1,ABB,Mar 2014,11.731107,25.126904,0.0
2,ABB,Mar 2015,13.630406,24.439701,0.0
3,ABB,Mar 2016,13.963275,21.338912,0.0
4,ABB,Mar 2017,13.709955,19.971161,0.0


In [36]:
# Get the most recent year's data for each company
latest_data = financials.sort_values('year').groupby('company_id').tail(1).copy()
latest_data.dropna(subset=['ROE_pct', 'Debt_to_Equity', 'OPM_pct'], inplace=True)

# Select features for clustering and scale them
features = ['ROE_pct', 'Debt_to_Equity', 'OPM_pct']
scaler = StandardScaler()
scaled_features = scaler.fit_transform(latest_data[features])

# Apply KMeans Clustering (k=5)
kmeans = KMeans(n_clusters=5, random_state=42)
latest_data['Cluster'] = kmeans.fit_predict(scaled_features)

# Label the clusters
cluster_mapping = {
    0: 'Value Cyclicals',
    1: 'High-Quality Growth',
    2: 'Defensive Dividend',
    3: 'Distressed / High Debt',
    4: 'Emerging Growth'
}
latest_data['Cluster_Name'] = latest_data['Cluster'].map(cluster_mapping)

# Save Cluster profiles
latest_data.to_csv('cluster_labels.csv', index=False)

# Visualise the Clusters using Plotly
fig = px.scatter(
    latest_data,
    x='ROE_pct',
    y='Debt_to_Equity',
    color='Cluster_Name',
    hover_data=['company_id', 'OPM_pct'],
    title="Nifty 100: Statistical Clustering by Financial Health",
    labels={'ROE_pct': 'Return on Equity (%)', 'Debt_to_Equity': 'Debt to Equity Ratio'}
)
fig.show()

In [37]:
# Example Screener Logic: Look for companies with high ROE, low Debt, and good Margins
screener_results = latest_data[
    (latest_data['ROE_pct'] > 15.0) &       # ROE greater than 15%
    (latest_data['Debt_to_Equity'] < 1.0) & # Low Debt
    (latest_data['OPM_pct'] > 10.0)         # Operating margin > 10%
].sort_values(by='ROE_pct', ascending=False)

print(f"Screener Found {len(screener_results)} matching companies.")
display(screener_results[['company_id', 'ROE_pct', 'Debt_to_Equity', 'OPM_pct']])

# Export Screener to Excel
screener_results.to_excel('screener_results.xlsx', index=False)

Screener Found 35 matching companies.


,company_id,ROE_pct,Debt_to_Equity,OPM_pct
226,BEL,4744.047619,0.511905,24.921058
439,HAL,3816.582915,0.618090,32.089135
1174,INDIGO,892.568306,0.018579,76.298909
779,NESTLEIND,117.754491,0.103293,23.829630
707,LT,77.665101,0.103457,13.506669
274,BRITANNIA,54.148693,0.523979,18.886040
1066,TCS,50.944314,0.088641,26.690688
78,ADANIPOWER,48.276741,0.802318,36.201863
322,COALINDIA,45.169830,0.078847,66.293808
1126,TRENT,36.307768,0.430924,15.927273


In [40]:
%%writefile app.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go

# Set page configuration
st.set_page_config(
    page_title="Nifty 100 Financial Intelligence Platform",
    page_icon="📊",
    layout="wide"
)

# Database Connection Helper
def get_db_connection():
    return sqlite3.connect('nifty100.db')

# Title & Description
st.title("📊 Nifty 100 Financial Intelligence Platform")
st.markdown("### Module 5: Enterprise Interactive Executive Dashboard")
st.write("---")

# 1. Sidebar - Data Loading & Filtering Engine
conn = get_db_connection()

# Fetch company list
try:
    companies_df = pd.read_sql("SELECT id, company_name FROM companies ORDER BY company_name", conn)
    company_dict = dict(zip(companies_df['company_name'], companies_df['id']))

    selected_company_name = st.sidebar.selectbox("Select Company Universe", list(company_dict.keys()))
    selected_id = company_dict[selected_company_name]
except Exception as e:
    st.sidebar.error("Error loading company list. Ensure Module 1 and 2 ran successfully.")
    st.stop()

# Fetch specific company details
comp_info = pd.read_sql(f"SELECT * FROM companies WHERE id = '{selected_id}'", conn).iloc[0]
pnl_df = pd.read_sql(f"SELECT * FROM profitandloss WHERE company_id = '{selected_id}' ORDER BY year", conn)
bs_df = pd.read_sql(f"SELECT * FROM balancesheet WHERE company_id = '{selected_id}' ORDER BY year", conn)
cf_df = pd.read_sql(f"SELECT * FROM cashflow WHERE company_id = '{selected_id}' ORDER BY year", conn)
pc_df = pd.read_sql(f"SELECT * FROM prosandcons WHERE company_id = '{selected_id}'", conn)

# Try fetching ratios if table exists
try:
    ratio_df = pd.read_sql(f"SELECT * FROM financial_ratios WHERE company_id = '{selected_id}' ORDER BY year", conn)
except:
    ratio_df = pd.DataFrame()

conn.close()

# 2. Company Profile Layout Header
col1, col2 = st.columns([1, 4])
with col1:
    if pd.notna(comp_info.get('company_logo')):
        st.image(comp_info['company_logo'], width=150)
with col2:
    st.subheader(comp_info['company_name'])
    st.caption(f"Ticker Symbol: {comp_info['id']} | [Official Website]({comp_info.get('website', '#')})")
    st.write(comp_info.get('about_company', 'No description available.'))

st.write("---")

# 3. Dynamic Key Metrics (KPI Grid)
if not pnl_df.empty:
    latest_pnl = pnl_df.iloc[-1]
    prev_pnl = pnl_df.iloc[-2] if len(pnl_df) > 1 else latest_pnl

    # Simple conversion logic for values safely
    def clean_val(val):
        try: return float(str(val).replace(',', ''))
        except: return 0.0

    sales_current = clean_val(latest_pnl.get('sales', 0))
    sales_prev = clean_val(prev_pnl.get('sales', 0))
    sales_delta = ((sales_current - sales_prev) / sales_prev * 100) if sales_prev != 0 else 0

    profit_current = clean_val(latest_pnl.get('net_profit', 0))
    profit_prev = clean_val(prev_pnl.get('net_profit', 0))
    profit_delta = ((profit_current - profit_prev) / profit_prev * 100) if profit_prev != 0 else 0

    st.markdown("#### Latest Financial Performance Metrics")
    kpi1, kpi2, kpi3, kpi4 = st.columns(4)
    kpi1.metric(label="Revenue (Sales) 🪙", value=f"₹{sales_current:,.2f} Cr", delta=f"{sales_delta:.2f}% YoY")
    kpi2.metric(label="Net Profit 📈", value=f"₹{profit_current:,.2f} Cr", delta=f"{profit_delta:.2f}% YoY")
    kpi3.metric(label="Book Value 📘", value=f"₹{comp_info.get('book_value', 'N/A')}")
    kpi4.metric(label="ROCE % ⚡", value=f"{comp_info.get('roce_percentage', 'N/A')}%")

st.write("---")

# 4. Multi-Tab Visualization Matrix
tab1, tab2, tab3, tab4 = st.tabs(["📊 Performance Trends", "📋 Financial Statements", "🔍 Qualitative Pros & Cons", "📈 Ratio Analysis"])

with tab1:
    st.subheader("Historical Trajectory Visualizations")
    if not pnl_df.empty:
        # P&L Charting (Sales vs Net Profit)
        fig_trend = go.Figure()
        fig_trend.add_trace(go.Bar(x=pnl_df['year'], y=pnl_df['sales'], name='Revenue / Sales', marker_color='#1f77b4'))
        fig_trend.add_trace(go.Scatter(x=pnl_df['year'], y=pnl_df['net_profit'], name='Net Profit', mode='lines+markers', line=dict(color='#ff7f0e', width=3)))
        fig_trend.update_layout(title="Revenue vs Net Profit Trend Lines", xaxis_title="Fiscal Year", yaxis_title="Amount (In ₹ Crores)", barmode='group')
        st.plotly_chart(fig_trend, use_container_width=True)

    if not cf_df.empty:
        # Cash Flow Breakup Charting
        fig_cf = px.line(cf_df, x='year', y=['operating_activity', 'investing_activity', 'financing_activity'],
                         labels={'value': 'Cash Flow (₹ Cr)', 'variable': 'Activities'},
                         title='Cash Flow Lifecycle Breakdown')
        st.plotly_chart(fig_cf, use_container_width=True)

with tab2:
    st.subheader("Structured Financial Data Tables")
    st.markdown("**Profit & Loss Summary**")
    st.dataframe(pnl_df, use_container_width=True)

    st.markdown("**Balance Sheet Position Statement**")
    st.dataframe(bs_df, use_container_width=True)

with tab3:
    st.subheader("Qualitative Sentiments / Fundamental Strength Check")
    if not pc_df.empty:
        p_col, c_col = st.columns(2)
        with p_col:
            st.success("🟢 PROS / Strengths")
            pros_list = pc_df['pros'].dropna().unique()
            for pro in pros_list:
                if str(pro).strip() and str(pro).upper() != "NULL":
                    st.write(f"- {pro}")
        with c_col:
            st.error("🔴 CONS / Risks")
            cons_list = pc_df['cons'].dropna().unique()
            for con in cons_list:
                if str(con).strip() and str(con).upper() != "NULL":
                    st.write(f"- {con}")
    else:
        st.info("No explicit Qualitative Pros or Cons documented for this entity.")

with tab4:
    st.subheader("Computed Engine Ratios")
    if not ratio_df.empty:
        st.dataframe(ratio_df[['year', 'OPM_pct', 'ROE_pct', 'Debt_to_Equity']], use_container_width=True)

        # Interactive Ratio Plots
        fig_ratio = px.line(ratio_df, x='year', y='ROE_pct', title='Return on Equity (ROE %) Over Time', markers=True)
        st.plotly_chart(fig_ratio, use_container_width=True)
    else:
        st.warning("Please compute and execute Module 2 to check advanced visual ratio analytical engines.")

Overwriting app.py


In [42]:
import pandas as pd
import sqlite3
import numpy as np

class FinancialAnalyticsEngine:
    def __init__(self, db_path='nifty100.db'):
        self.db_path = db_path

    def get_connection(self):
        return sqlite3.connect(self.db_path)

    # MODULE 3: ADVANCED INVESTMENT SCREENER ENGINE

    def execute_screener(self, min_roe=15.0, max_debt_to_equity=1.5, min_opm=10.0):
        """
        Filters the universe based on multi-parameter fundamental constraints
        and scores companies using an aggregated rank-based health score.
        """
        conn = self.get_connection()

        # Comprehensive query pulling latest available financial parameters
        query = """
        SELECT
            r.company_id,
            c.company_name,
            r.year,
            r.ROE_pct,
            r.Debt_to_Equity,
            r.OPM_pct,
            r.net_profit,
            r.sales
        FROM financial_ratios r
        JOIN companies c ON r.company_id = c.id
        WHERE r.year = (SELECT MAX(year) FROM financial_ratios WHERE company_id = r.company_id)
        """
        df = pd.read_sql(query, conn)
        conn.close()

        # Clean null values safely
        df.dropna(subset=['ROE_pct', 'Debt_to_Equity', 'OPM_pct'], inplace=True)

        # Apply strict parameters screening criteria
        filtered_df = df[
            (df['ROE_pct'] >= min_roe) &
            (df['Debt_to_Equity'] <= max_debt_to_equity) &
            (df['OPM_pct'] >= min_opm)
        ].copy()

        if filtered_df.empty:
            print("⚠️ No companies matched the screening criteria.")
            return filtered_df

        # Programmatic Composite Scoring (Percentile Rank aggregation)
        filtered_df['roe_rank'] = filtered_df['ROE_pct'].rank(pct=True)
        filtered_df['margin_rank'] = filtered_df['OPM_pct'].rank(pct=True)
        filtered_df['debt_rank'] = filtered_df['Debt_to_Equity'].rank(pct=True, ascending=False) # Lower debt is better

        # Calculate final proprietary Financial Health Score out of 100
        filtered_df['Financial_Health_Score'] = ((filtered_df['roe_rank'] + filtered_df['margin_rank'] + filtered_df['debt_rank']) / 3) * 100
        filtered_df['Financial_Health_Score'] = filtered_df['Financial_Health_Score'].round(2)

        # Sort by best financial score descending
        screener_results = filtered_df.sort_values(by='Financial_Health_Score', ascending=False)

        # Persist results to a specialized table in SQLite
        conn = self.get_connection()
        screener_results[['company_id', 'company_name', 'year', 'ROE_pct', 'Debt_to_Equity', 'OPM_pct', 'Financial_Health_Score']].to_sql(
            'screener_output', conn, if_exists='replace', index=False
        )
        conn.close()

        return screener_results[['company_id', 'company_name', 'ROE_pct', 'Debt_to_Equity', 'OPM_pct', 'Financial_Health_Score']]


    # MODULE 4: COHORT PEER BENCHMARKING ENGINE

    def generate_peer_analysis(self, target_company_id):
        """
        Uses the statistical clusters created in Module 10 to establish
        a synthetic peer cohort and runs relative metric benchmarking.
        """
        target_company_id = target_company_id.strip().upper()

        # Read cluster profiles generated during Module 10 execution
        try:
            clusters_df = pd.read_csv('cluster_labels.csv')
        except FileNotFoundError:
            print("❌ Error: 'cluster_labels.csv' from Module 10 not found. Please run Module 10 first.")
            return None

        if target_company_id not in clusters_df['company_id'].values:
            print(f"⚠️ Target Ticker '{target_company_id}' not found in the baseline data registry.")
            return None

        # Identify target company cluster profile
        target_cluster = clusters_df[clusters_df['company_id'] == target_company_id]['Cluster_Name'].values[0]

        # Fetch all peers sharing the identical strategy/financial cluster
        peer_cohort = clusters_df[clusters_df['Cluster_Name'] == target_cluster].copy()

        print(f"\n📊 Peer Cohort Analysis for: {target_company_id} [Cluster: {target_cluster}]")
        print(f"Found {len(peer_cohort)} matching peers inside this financial profile archetype.\n")

        # Benchmarking calculations: Calculate cross-sectional averages
        cohort_avg_roe = peer_cohort['ROE_pct'].mean()
        cohort_avg_debt = peer_cohort['Debt_to_Equity'].mean()
        cohort_avg_opm = peer_cohort['OPM_pct'].mean()

        # Compute variances from peer group average
        target_metrics = peer_cohort[peer_cohort['company_id'] == target_company_id].iloc[0]

        benchmarks = {
            'Metric': ['Return on Equity (ROE %)', 'Debt to Equity Ratio', 'Operating Margin (OPM %)'],
            'Target Company Value': [round(target_metrics['ROE_pct'], 2), round(target_metrics['Debt_to_Equity'], 2), round(target_metrics['OPM_pct'], 2)],
            'Cohort Peer Average': [round(cohort_avg_roe, 2), round(cohort_avg_debt, 2), round(cohort_avg_opm, 2)],
        }

        bench_df = pd.DataFrame(benchmarks)
        bench_df['Variance (%)'] = ((bench_df['Target Company Value'] - bench_df['Cohort Peer Average']) / bench_df['Cohort Peer Average'] * 100).round(2)

        return bench_df


# PIPELINE EXECUTION GATEWAY
# ==========================================
# Initialize the integrated engine
analytics_engine = FinancialAnalyticsEngine()

print("--- Executing Module 3: Enterprise Stock Screener ---")
# Run screener: e.g., Filter companies with ROE > 12%, Debt-to-Equity < 1.2, Margin > 8%
screener_matches = analytics_engine.execute_screener(min_roe=12.0, max_debt_to_equity=1.2, min_opm=8.0)
display(screener_matches.head(10))

print("\n--- Executing Module 4: Sector/Cohort Peer Benchmarking ---")
# Run peer benchmarking relative engine for a choice ticker (e.g., 'HDFCBANK', 'INFY', or 'WIPRO')
# It will find its peers from the K-Means cluster grouping automatically
peer_benchmarking_report = analytics_engine.generate_peer_analysis(target_company_id='INFY')

if peer_benchmarking_report is not None:
    display(peer_benchmarking_report)

--- Executing Module 3: Enterprise Stock Screener ---


,company_id,company_name,ROE_pct,Debt_to_Equity,OPM_pct,Financial_Health_Score
98,INDIGO,Interglobe Aviation Ltd,892.568306,0.018579,76.298909,90.15
51,IRCTC,Indian Railway Catering & Tourism Corporation Ltd,34.396285,0.018576,34.332553,78.79
53,ITC,ITC Ltd\n,27.851074,0.004067,37.017752,78.03
27,COALINDIA,Coal India Ltd,45.169830,0.078847,66.293808,75.00
46,ICICIGI,ICICI Lombard General Insurance Company Ltd,15.723064,0.002868,87.421292,71.97
42,HEROMOTOCO,Hero MotoCorp Ltd\n,21.142437,0.034239,86.146762,71.21
15,BAJAJHLDNG,Bajaj Holdings & Investment Ltd,13.576788,0.001161,91.774383,70.45
44,HINDUNILVR,Hindustan Unilever Ltd\n,20.074974,0.028974,76.316725,69.70
0,ABB,Abbott India Ltd,32.468235,0.022438,24.841853,69.70
91,TCS,Tata Consultancy Services Ltd,50.944314,0.088641,26.690688,67.42



--- Executing Module 4: Sector/Cohort Peer Benchmarking ---

📊 Peer Cohort Analysis for: INFY [Cluster: Value Cyclicals]
Found 61 matching peers inside this financial profile archetype.



,Metric,Target Company Value,Cohort Peer Average,Variance (%)
0,Return on Equity (ROE %),29.79,21.85,36.34
1,Debt to Equity Ratio,0.09,0.65,-86.15
2,Operating Margin (OPM %),23.70,19.78,19.82


In [43]:
%%writefile api.py
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
import sqlite3
import pandas as pd
import os

# Initialize FastAPI Application
app = FastAPI(
    title="Nifty 100 Financial Intelligence Platform API Engine",
    description="Enterprise REST Core Layer for financial data queries, ratio inspection, and cohort benchmarking.",
    version="1.0.0"
)

# Enable CORS Middleware for cross-origin presentation layers
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

DB_PATH = "nifty100.db"

def query_db(sql_query: str, params=()):
    """Helper method to connect, execute, and securely fetch tabular data as a Dataframe."""
    if not os.path.exists(DB_PATH):
        raise HTTPException(status_code=500, detail="Database file 'nifty100.db' missing. Execute Module 1 and Module 2 first.")
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql_query, conn, params=params)
        conn.close()
        return df
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Database Execution Fault: {str(e)}")


# ENDPOINT 1: CORE ENGINE HEALTHCHECK
# ==========================================
@app.get("/", tags=["System Diagnostics"])
def read_root():
    return {
        "status": "ONLINE",
        "service": "Nifty 100 Financial Intelligence API Core",
        "documentation_route": "/docs"
    }

# ENDPOINT 2: RETRIEVE ALL REGISTERED COMPANIES
# ==========================================
@app.get("/api/v1/companies", tags=["Market Universe"])
def get_all_companies():
    """Returns baseline data for all 92 tracked Nifty 100 corporate entities."""
    query = "SELECT id, company_name, website, book_value, roce_percentage FROM companies ORDER BY company_name"
    df = query_db(query)
    return df.to_dict(orient="records")

# ENDPOINT 3: RETRIEVE DETAILED COMPANY PROFILE
# ==========================================
@app.get("/api/v1/companies/{company_id}", tags=["Market Universe"])
def get_company_profile(company_id: str):
    """Fetches full granular registry profile metrics for a specific corporate identifier (Ticker Symbol)."""
    query = "SELECT * FROM companies WHERE UPPER(id) = ?"
    df = query_db(query, (company_id.strip().upper(),))

    if df.empty:
        raise HTTPException(status_code=404, detail=f"Ticker target '{company_id}' not found in registry matrix.")
    return df.iloc[0].to_dict()

# ENDPOINT 4: HISTORICAL RATIOS & PERFORMANCE EXPORTER
# ==========================================
@app.get("/api/v1/financials/{company_id}", tags=["Financial Statements"])
def get_historical_financials(company_id: str):
    """Extracts structural time-series financial ratios and core balance sheet/profit-loss records."""
    ticker = company_id.strip().upper()

    # Query performance tables
    pnl = query_db("SELECT * FROM profitandloss WHERE UPPER(company_id) = ? ORDER BY year", (ticker,))
    ratios = query_db("SELECT year, OPM_pct, ROE_pct, Debt_to_Equity FROM financial_ratios WHERE UPPER(company_id) = ? ORDER BY year", (ticker,))
    pros_cons = query_db("SELECT pros, cons FROM prosandcons WHERE UPPER(company_id) = ?", (ticker,))

    if pnl.empty:
        raise HTTPException(status_code=404, detail=f"No underlying financial statements located for ticker: {company_id}")

    return {
        "ticker": ticker,
        "historical_records_count": len(pnl),
        "profit_and_loss_trajectory": pnl.to_dict(orient="records"),
        "computed_financial_ratios": ratios.to_dict(orient="records") if not ratios.empty else [],
        "qualitative_sentiments": pros_cons.to_dict(orient="records") if not pros_cons.empty else []
    }

# ENDPOINT 5: LIVE SYSTEM INVESTMENT SCREENER EXTRACTOR
# ==========================================
@app.get("/api/v1/screener", tags=["Analytics & Alpha Metrics"])
def get_screener_results():
    """Exposes current alpha opportunities isolated by the Module 3 Advanced Screening Engine."""
    if not os.path.exists(DB_PATH):
        raise HTTPException(status_code=500, detail="Database context missing.")

    try:
        df = query_db("SELECT * FROM screener_output ORDER BY Financial_Health_Score DESC")
        return {
            "total_matches": len(df),
            "screened_entities": df.to_dict(orient="records")
        }
    except Exception:
        raise HTTPException(status_code=404, detail="Screener output table empty or missing. Please execute Module 3 first.")

# ENDPOINT 6: COHORT PEER BENCHMARK VARIANCE CALCULATOR
# ==========================================
@app.get("/api/v1/peers/{company_id}", tags=["Analytics & Alpha Metrics"])
def get_peer_benchmarking(company_id: str):
    """Computes cross-sectional cohort variances using statistical clusters generated via Module 10."""
    ticker = company_id.strip().upper()

    if not os.path.exists('cluster_labels.csv'):
        raise HTTPException(status_code=500, detail="Statistical Cluster labels document missing. Execute Module 10 first.")

    clusters_df = pd.read_csv('cluster_labels.csv')
    if ticker not in clusters_df['company_id'].values:
        raise HTTPException(status_code=404, detail=f"Ticker symbol '{ticker}' not evaluated within active classification clusters.")

    # Isolate cluster peers
    target_cluster = clusters_df[clusters_df['company_id'] == ticker]['Cluster_Name'].values[0]
    peer_cohort = clusters_df[clusters_df['Cluster_Name'] == target_cluster]

    target_metrics = peer_cohort[peer_cohort['company_id'] == ticker].iloc[0]

    # Calculate cohort metrics
    response_payload = {
        "target_ticker": ticker,
        "financial_archetype_cluster": target_cluster,
        "cohort_peer_count": len(peer_cohort),
        "benchmarks": [
            {
                "metric": "Return on Equity (ROE %)",
                "target_value": float(target_metrics['ROE_pct']),
                "cohort_average": float(peer_cohort['ROE_pct'].mean())
            },
            {
                "metric": "Debt to Equity Ratio",
                "target_value": float(target_metrics['Debt_to_Equity']),
                "cohort_average": float(peer_cohort['Debt_to_Equity'].mean())
            },
            {
                "metric": "Operating Margin (OPM %)",
                "target_value": float(target_metrics['OPM_pct']),
                "cohort_average": float(peer_cohort['OPM_pct'].mean())
            }
        ]
    }
    return response_payload

Overwriting api.py


In [44]:
# Print out your local IP address which acts as your tunnel access verification key
!curl ipv4.icanhazip.com
# Launch the ASGI production server in the background and bind it to a localtunnel proxy
!uvicorn api:app --host 0.0.0.0 --port 8000 & npx localtunnel --port 8000

34.87.4.111
⠙⠹⠸⠼⠴⠦INFO:     Started server process [28218]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
your url is: https://plain-windows-carry.loca.lt
INFO:     103.82.97.10:0 - "GET / HTTP/1.1" 200 OK
INFO:     103.82.97.10:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [28218]
^C
